# LeetCode 1615 — Maximal Network Rank bằng Python

Link đề: https://leetcode.com/problems/maximal-network-rank/ · Độ khó: **Medium** · Chủ đề: Graph

Skeleton LeetCode:

```python
class Solution:
    def maximalNetworkRank(self, n: int, roads: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biểu diễn đồ thị vô hướng bằng **mảng bậc** (degree), **tập kề** (adjacency set) và **ma trận kề**.
2. Hiểu công thức cốt lõi: `rank(a, b) = deg[a] + deg[b] − (1 nếu a, b nối trực tiếp, ngược lại 0)` và vì sao phải trừ 1.
3. Đi từ vét cạn `O(n²·m)` → duyệt cặp `O(n² + m)` → sắp xếp + cắt tỉa → phân tích trường hợp `O(n + m)`.
4. Tự tìm **phản ví dụ** cho các heuristic sai (chỉ lấy 2 thành phố bậc cao nhất, chỉ xét các cặp có đường nối…).
5. Viết bộ test ngẫu nhiên có seed cố định đối chiếu nhiều cách, đúng ràng buộc đề.

## 2. Đề bài & ví dụ

Có `n` thành phố đánh số `0..n−1` và danh sách `roads`, mỗi `roads[i] = [a, b]` là một **đường hai chiều** nối `a` và `b`.

**Network rank** của **hai thành phố khác nhau** là tổng số đường nối **trực tiếp** với **ít nhất một** trong hai thành phố. Nếu một đường nối cả hai thành phố thì chỉ đếm **một lần**.

**Maximal network rank** là giá trị network rank **lớn nhất** trên mọi cặp thành phố khác nhau. Cho `n` và `roads`, trả về maximal network rank.

**Ví dụ 1:** `n = 4, roads = [[0,1],[0,3],[1,2],[1,3]]` → `4`. Cặp (0, 1): các đường chạm 0 hoặc 1 là `[0,1],[0,3],[1,2],[1,3]` — 4 đường, đường `[0,1]` chỉ đếm một lần.

**Ví dụ 2:** `n = 5, roads = [[0,1],[0,3],[1,2],[1,3],[2,3],[2,4]]` → `5`. Có 5 đường chạm thành phố 1 hoặc 2.

**Ví dụ 3:** `n = 8, roads = [[0,1],[1,2],[2,3],[2,4],[5,6],[5,7]]` → `5`. Cặp (2, 5) có rank 5. Lưu ý: các thành phố **không cần** liên thông với nhau.

**Ràng buộc:**
- `2 ≤ n ≤ 100`
- `0 ≤ roads.length ≤ n·(n−1)/2`
- `roads[i].length == 2`, `0 ≤ a, b ≤ n−1`, `a ≠ b`
- Mỗi cặp thành phố có **nhiều nhất một** đường nối.

In [1]:
import random
import time
from itertools import combinations
from collections import defaultdict
from typing import List

examples = [
    (4, [[0, 1], [0, 3], [1, 2], [1, 3]], 4),
    (5, [[0, 1], [0, 3], [1, 2], [1, 3], [2, 3], [2, 4]], 5),
    (8, [[0, 1], [1, 2], [2, 3], [2, 4], [5, 6], [5, 7]], 5),
]
for n, roads, exp in examples:
    print(f"n={n}, roads={roads} -> kỳ vọng {exp}")

n=4, roads=[[0, 1], [0, 3], [1, 2], [1, 3]] -> kỳ vọng 4
n=5, roads=[[0, 1], [0, 3], [1, 2], [1, 3], [2, 3], [2, 4]] -> kỳ vọng 5
n=8, roads=[[0, 1], [1, 2], [2, 3], [2, 4], [5, 6], [5, 7]] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1 Bậc của đỉnh (degree)

Trong đồ thị vô hướng, **bậc** của thành phố `v` là số đường nối với `v`. Mỗi đường `[a, b]` làm tăng bậc của **cả** `a` và `b` thêm 1, nên tổng các bậc = `2·m` (m = số đường).

In [2]:
def degrees(n, roads):
    deg = [0] * n
    for a, b in roads:
        deg[a] += 1
        deg[b] += 1
    return deg

n, roads, _ = examples[0]
deg = degrees(n, roads)
print("deg =", deg, "| tổng bậc =", sum(deg), "= 2*m =", 2 * len(roads))

deg = [2, 3, 1, 2] | tổng bậc = 8 = 2*m = 8


### 3.2 Tập kề (adjacency set) và ma trận kề

Để hỏi nhanh "`a` và `b` có nối trực tiếp không?":
- **Tập kề:** `adj[v]` là `set` các hàng xóm của `v`; `b in adj[a]` tốn `O(1)` trung bình.
- **Ma trận kề:** `g[a][b] = 1` nếu có đường; truy cập `O(1)`, tốn `O(n²)` bộ nhớ — với `n ≤ 100` chỉ là 10.000 ô, rất nhỏ.
- **Tập cạnh chuẩn hóa:** lưu `(min(a,b), max(a,b))` vào một `set` cũng được.

In [3]:
adj = [set() for _ in range(n)]
g = [[0] * n for _ in range(n)]
edges = set()
for a, b in roads:
    adj[a].add(b); adj[b].add(a)
    g[a][b] = g[b][a] = 1
    edges.add((min(a, b), max(a, b)))
print("adj =", adj)
print("g =", g)
print("0-1 nối?", 1 in adj[0], g[0][1], (0, 1) in edges, "| 0-2 nối?", 2 in adj[0], g[0][2], (0, 2) in edges)

adj = [{1, 3}, {0, 2, 3}, {1}, {0, 1}]
g = [[0, 1, 0, 1], [1, 0, 1, 1], [0, 1, 0, 0], [1, 1, 0, 0]]
0-1 nối? True 1 True | 0-2 nối? False 0 False


### 3.3 Duyệt mọi cặp khác nhau

Có `n·(n−1)/2` cặp không thứ tự. Hai cách viết: vòng lặp lồng `for i in range(n): for j in range(i + 1, n)` hoặc `itertools.combinations(range(n), 2)`.

In [4]:
print(list(combinations(range(4), 2)))
pairs = [(i, j) for i in range(4) for j in range(i + 1, 4)]
print(pairs, "| số cặp:", len(pairs), "= 4*3/2")

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)] | số cặp: 6 = 4*3/2


### 3.4 Sắp xếp theo khóa và `max` với generator

`sorted(range(n), key=lambda v: -deg[v])` sắp các thành phố theo bậc giảm dần. `max(biểu_thức for ...)` lấy giá trị lớn nhất của một dãy sinh ra tại chỗ, không cần tạo list; thêm `default=0` nếu dãy có thể rỗng.

In [5]:
order = sorted(range(n), key=lambda v: -deg[v])
print("thứ tự theo bậc giảm:", order, [deg[v] for v in order])
print(max((x * x for x in [3, -5, 2]), default=0), max((x for x in []), default=0))

thứ tự theo bậc giảm: [1, 0, 3, 2] [3, 2, 2, 1]
25 0


## 4. Ý tưởng nền tảng / trực giác

Tính bằng tay **Ví dụ 1**: `roads = [[0,1],[0,3],[1,2],[1,3]]`.

| Thành phố | Hàng xóm | Bậc |
|---|---|---|
| 0 | 1, 3 | 2 |
| 1 | 0, 2, 3 | 3 |
| 2 | 1 | 1 |
| 3 | 0, 1 | 2 |

Cặp (0, 1): `deg[0] + deg[1] = 2 + 3 = 5`. Nhưng đường `[0,1]` đã bị đếm **hai lần** (một lần trong bậc của 0, một lần trong bậc của 1) ⇒ trừ 1 ⇒ `4`.

Cặp (1, 2): `3 + 1 − 1 = 3`. Cặp (1, 3): `3 + 2 − 1 = 4`. Cặp (0, 2): `2 + 1 − 0 = 3` (không nối). Cặp (0, 3): `2 + 2 − 1 = 3`. Cặp (2, 3): `1 + 2 − 0 = 3`. ⇒ Lớn nhất `4`.

**Quan sát:** network rank chỉ phụ thuộc vào **bậc** của hai thành phố và việc **chúng có nối nhau không**.

In [6]:
n, roads, _ = examples[0]
deg = degrees(n, roads)
E = {(min(a, b), max(a, b)) for a, b in roads}
for i, j in combinations(range(n), 2):
    conn = (i, j) in E
    print(f"({i},{j}): {deg[i]} + {deg[j]} - {int(conn)} = {deg[i] + deg[j] - conn}")

(0,1): 2 + 3 - 1 = 4
(0,2): 2 + 1 - 0 = 3
(0,3): 2 + 2 - 1 = 3
(1,2): 3 + 1 - 1 = 3
(1,3): 3 + 2 - 1 = 4
(2,3): 1 + 2 - 0 = 3


## 5. Cách 1 — Brute Force / baseline: đếm trực tiếp theo định nghĩa

**Ý tưởng:** với mỗi cặp `(i, j)`, duyệt **toàn bộ** danh sách đường, đếm đường nào chạm `i` **hoặc** `j`. Chữ "hoặc" tự động đếm đường `[i, j]` đúng một lần — không cần nghĩ tới việc trừ.

In [7]:
def rank_bruteforce(n, roads):
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            cnt = 0
            for a, b in roads:
                if a == i or a == j or b == i or b == j:
                    cnt += 1
            best = max(best, cnt)
    return best

for n, roads, exp in examples:
    got = rank_bruteforce(n, roads)
    print(got, "kỳ vọng", exp)
    assert got == exp

4 kỳ vọng 4
5 kỳ vọng 5
5 kỳ vọng 5


**Độ phức tạp:** `O(n² · m)` thời gian, `O(1)` bộ nhớ phụ. Với `n = 100`, `m` tối đa `4950`: `4950 · 4950 ≈ 2,45·10⁷` phép kiểm tra — chậm trong Python (vài giây).

**Nút thắt:** với mỗi cặp ta lại đọc lại toàn bộ danh sách đường. Nhưng thông tin "bao nhiêu đường chạm `i`" là **cố định** — chính là `deg[i]` — có thể tính **một lần** trước.

In [8]:
rng = random.Random(1)
n_big = 100
all_pairs = list(combinations(range(n_big), 2))
roads_big = [list(p) for p in rng.sample(all_pairs, 1500)]
t = time.perf_counter(); r = rank_bruteforce(n_big, roads_big); dt = time.perf_counter() - t
print(f"brute force n=100, m=1500 -> {r} trong {dt:.2f} s")

brute force n=100, m=1500 -> 84 trong 0.33 s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (bao hàm – loại trừ):** gọi `A` = tập đường chạm `i`, `B` = tập đường chạm `j`.
`|A ∪ B| = |A| + |B| − |A ∩ B|`, với `|A| = deg[i]`, `|B| = deg[j]`, và `A ∩ B` = các đường chạm **cả** `i` và `j` — chỉ có thể là đường `[i, j]` (vì giữa hai thành phố có nhiều nhất một đường). Vậy:

`rank(i, j) = deg[i] + deg[j] − connected(i, j)`, với `connected ∈ {0, 1}`.

**Quy tắc "nếu… thì…":**
- Nếu `i`, `j` **không nối** thì `rank = deg[i] + deg[j]`.
- Nếu `i`, `j` **nối** thì `rank = deg[i] + deg[j] − 1`.
- Nếu `deg[i] + deg[j] ≤ ans` hiện tại thì cặp này (và các cặp có tổng bậc nhỏ hơn) **không thể** cải thiện ⇒ cắt tỉa.

Từ đó: tiền xử lý bậc + kiểm tra kề `O(1)` ⇒ mỗi cặp tốn `O(1)` ⇒ tổng `O(n² + m)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng bậc + ma trận kề, duyệt mọi cặp

**Giải thích từng đoạn:**
1. Một lượt qua `roads`: tăng `deg[a]`, `deg[b]`, đặt `g[a][b] = g[b][a] = 1`.
2. Duyệt mọi cặp `i < j`, tính `deg[i] + deg[j] − g[i][j]`, giữ giá trị lớn nhất.

In [9]:
def rank_pairs(n, roads):
    deg = [0] * n
    g = [[0] * n for _ in range(n)]
    for a, b in roads:
        deg[a] += 1; deg[b] += 1
        g[a][b] = g[b][a] = 1
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            best = max(best, deg[i] + deg[j] - g[i][j])
    return best

for n, roads, exp in examples:
    assert rank_pairs(n, roads) == exp
t = time.perf_counter(); r = rank_pairs(n_big, roads_big); dt = time.perf_counter() - t
print(f"Cách 2 trên n=100, m=1500 -> {r} trong {dt*1000:.1f} ms")

Cách 2 trên n=100, m=1500 -> 84 trong 0.3 ms


**Code nộp LeetCode (Cách 2):**

In [10]:
class SolutionPairs:
    def maximalNetworkRank(self, n: int, roads: List[List[int]]) -> int:
        deg = [0] * n
        adj = [set() for _ in range(n)]
        for a, b in roads:
            deg[a] += 1
            deg[b] += 1
            adj[a].add(b)
            adj[b].add(a)
        return max(deg[i] + deg[j] - (j in adj[i])
                   for i in range(n) for j in range(i + 1, n))

print([SolutionPairs().maximalNetworkRank(n, r) for n, r, _ in examples])

[4, 5, 5]


Ở đây `(j in adj[i])` là `bool`, mà trong Python `True == 1`, `False == 0` nên trừ trực tiếp được. Vì `n ≥ 2` nên luôn có ít nhất một cặp, `max` không bao giờ nhận dãy rỗng.

**Độ phức tạp Cách 2:** Time `O(n² + m)`, Space `O(n + m)` (tập kề) hoặc `O(n²)` (ma trận).

### 7.2 Cách 3 — Sắp xếp theo bậc giảm dần + cắt tỉa sớm

**Giải thích từng đoạn:**
1. Sắp thành phố theo bậc giảm: `order`.
2. Duyệt `i` theo `order`, rồi `j` sau `i`. Vì bậc giảm dần, `deg[order[i]] + deg[order[j]]` giảm dần khi `j` tăng.
3. Nếu `deg[u] + deg[v] ≤ best` thì mọi `j` phía sau cũng không hơn ⇒ `break` vòng trong. Nếu ngay cả `deg[u] + deg[order[i+1]] ≤ best` thì dừng hẳn.
4. Cặp chưa nối có `rank = tổng bậc` — đó là cận trên của mọi cặp sau, nên có thể dừng vòng trong ngay sau khi gặp nó.

Trong thực tế số cặp phải xét rất ít, dù độ phức tạp tệ nhất vẫn `O(n²)`.

In [11]:
def rank_sorted(n, roads, verbose=False):
    deg = [0] * n
    adj = [set() for _ in range(n)]
    for a, b in roads:
        deg[a] += 1; deg[b] += 1
        adj[a].add(b); adj[b].add(a)
    order = sorted(range(n), key=lambda v: -deg[v])
    best, checked = 0, 0
    for i in range(n):
        u = order[i]
        if i + 1 < n and deg[u] + deg[order[i + 1]] <= best:
            break                                   # không cặp nào còn lại có thể hơn
        for j in range(i + 1, n):
            v = order[j]
            s = deg[u] + deg[v]
            if s <= best:
                break                               # các j sau còn nhỏ hơn
            checked += 1
            if v in adj[u]:
                best = max(best, s - 1)
            else:
                best = s                            # s > best và là cận trên cho các j sau
                break
    if verbose:
        print(f"  số cặp phải xét: {checked} / {n*(n-1)//2}")
    return best

for n, roads, exp in examples:
    assert rank_sorted(n, roads, verbose=True) == exp
print("n=100:", rank_sorted(n_big, roads_big, verbose=True), "| Cách 2:", rank_pairs(n_big, roads_big))

  số cặp phải xét: 2 / 6
  số cặp phải xét: 3 / 10
  số cặp phải xét: 2 / 28
  số cặp phải xét: 1 / 4950
n=100: 84 | Cách 2: 84


**Code nộp LeetCode (Cách 3):**

In [12]:
class SolutionSorted:
    def maximalNetworkRank(self, n: int, roads: List[List[int]]) -> int:
        deg = [0] * n
        adj = [set() for _ in range(n)]
        for a, b in roads:
            deg[a] += 1; deg[b] += 1
            adj[a].add(b); adj[b].add(a)
        order = sorted(range(n), key=lambda v: -deg[v])
        best = 0
        for i in range(n - 1):
            u = order[i]
            if deg[u] + deg[order[i + 1]] <= best:
                break
            for j in range(i + 1, n):
                v = order[j]
                s = deg[u] + deg[v]
                if s <= best:
                    break
                if v in adj[u]:
                    best = max(best, s - 1)
                else:
                    best = s
                    break
        return best

print([SolutionSorted().maximalNetworkRank(n, r) for n, r, _ in examples])

[4, 5, 5]


**Độ phức tạp Cách 3:** Time `O(n log n + m)` cho tiền xử lý + số cặp được xét (tệ nhất `O(n²)`, ví dụ đồ thị đầy đủ). Space `O(n + m)`.

### 7.3 Cách 4 — Phân tích trường hợp theo bậc lớn nhất và nhì, `O(n + m)`

Gọi `d1` = bậc lớn nhất, `d2` = bậc lớn thứ nhì **khác** `d1` (giá trị bậc, không phải thành phố). Gọi `M` = tập thành phố có bậc `d1`, `k = |M|`.

**Trường hợp A — `k ≥ 2`:** đáp án là `2·d1` hoặc `2·d1 − 1`.
- Chỉ cặp nằm trong `M` mới đạt được `≥ 2·d1 − 1` (cặp có thành phố bậc `< d1` cho tối đa `d1 + (d1 − 1) = 2·d1 − 1`, không hơn).
- Có `k·(k−1)/2` cặp trong `M`. Đếm số đường có **cả hai đầu** trong `M`: `e_M`. Nếu `e_M < k·(k−1)/2` thì tồn tại cặp không nối ⇒ `2·d1`; ngược lại mọi cặp trong `M` đều nối ⇒ `2·d1 − 1`.

**Trường hợp B — `k = 1`:** gọi `u` là thành phố duy nhất bậc `d1`, `S` = tập thành phố bậc `d2`.
- Cặp tốt nhất phải chứa `u` (vì cặp không chứa `u` cho tối đa `2·d2 ≤ d1 + d2 − 1`).
- Nếu có `v ∈ S` **không** nối `u` thì đáp án `d1 + d2`, ngược lại `d1 + d2 − 1`. Đếm số đường giữa `u` và `S`: `e_uS`; so với `|S|`.

Mọi việc chỉ là đếm trên danh sách đường ⇒ `O(n + m)`.

In [13]:
def rank_linear(n, roads, verbose=False):
    deg = [0] * n
    for a, b in roads:
        deg[a] += 1; deg[b] += 1
    d1 = max(deg)
    M = [v for v in range(n) if deg[v] == d1]
    k = len(M)
    if k >= 2:
        inM = [deg[v] == d1 for v in range(n)]
        e_M = sum(1 for a, b in roads if inM[a] and inM[b])
        if verbose:
            print(f"  TH A: d1={d1}, k={k}, số cặp trong M={k*(k-1)//2}, số đường trong M={e_M}")
        return 2 * d1 if e_M < k * (k - 1) // 2 else 2 * d1 - 1
    u = M[0]
    d2 = max(deg[v] for v in range(n) if v != u)
    S = {v for v in range(n) if v != u and deg[v] == d2}
    e_uS = sum(1 for a, b in roads if (a == u and b in S) or (b == u and a in S))
    if verbose:
        print(f"  TH B: u={u}, d1={d1}, d2={d2}, |S|={len(S)}, số đường u–S={e_uS}")
    return d1 + d2 if e_uS < len(S) else d1 + d2 - 1

for n, roads, exp in examples:
    got = rank_linear(n, roads, verbose=True)
    print("  ->", got, "kỳ vọng", exp); assert got == exp

  TH B: u=1, d1=3, d2=2, |S|=2, số đường u–S=2
  -> 4 kỳ vọng 4
  TH A: d1=3, k=3, số cặp trong M=3, số đường trong M=3
  -> 5 kỳ vọng 5
  TH B: u=2, d1=3, d2=2, |S|=2, số đường u–S=1
  -> 5 kỳ vọng 5


**Code nộp LeetCode (Cách 4):**

In [14]:
class SolutionLinear:
    def maximalNetworkRank(self, n: int, roads: List[List[int]]) -> int:
        deg = [0] * n
        for a, b in roads:
            deg[a] += 1
            deg[b] += 1
        d1 = max(deg)
        M = [v for v in range(n) if deg[v] == d1]
        k = len(M)
        if k >= 2:
            e = sum(1 for a, b in roads if deg[a] == d1 and deg[b] == d1)
            return 2 * d1 if e < k * (k - 1) // 2 else 2 * d1 - 1
        u = M[0]
        d2 = max(deg[v] for v in range(n) if v != u)
        S = {v for v in range(n) if v != u and deg[v] == d2}
        e = sum(1 for a, b in roads if (a == u and b in S) or (b == u and a in S))
        return d1 + d2 if e < len(S) else d1 + d2 - 1

print([SolutionLinear().maximalNetworkRank(n, r) for n, r, _ in examples])

[4, 5, 5]


**Độ phức tạp Cách 4:** Time `O(n + m)`, Space `O(n)`. Với `n ≤ 100`, Cách 2 đã đủ nhanh; Cách 4 có giá trị khi `n` lớn (ví dụ `n = 10⁵`, `m = 10⁵`) — khi đó `O(n²)` không chạy nổi.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2**: `deg = [2, 3, 3, 3, 1]`.

| cặp | deg[i] | deg[j] | nối? | rank | best |
|---|---|---|---|---|---|
| (0,1) | 2 | 3 | có | 4 | 4 |
| (0,2) | 2 | 3 | không | 5 | 5 |
| (0,3) | 2 | 3 | có | 4 | 5 |
| (1,2) | 3 | 3 | có | 5 | 5 |
| … | | | | | 5 |

Hàm `trace_pairs` in toàn bộ:

In [15]:
def trace_pairs(n, roads):
    deg = degrees(n, roads)
    adj = [set() for _ in range(n)]
    for a, b in roads:
        adj[a].add(b); adj[b].add(a)
    print("deg =", deg)
    best, arg = 0, None
    for i in range(n):
        for j in range(i + 1, n):
            c = int(j in adj[i])
            r = deg[i] + deg[j] - c
            mark = ""
            if r > best:
                best, arg = r, (i, j); mark = "  <- cập nhật best"
            print(f"cặp ({i},{j}): {deg[i]} + {deg[j]} - {c} = {r} | best = {best}{mark}")
    print("Kết quả:", best, "đạt tại", arg)
    return best

assert trace_pairs(*examples[1][:2]) == 5

deg = [2, 3, 3, 3, 1]
cặp (0,1): 2 + 3 - 1 = 4 | best = 4  <- cập nhật best
cặp (0,2): 2 + 3 - 0 = 5 | best = 5  <- cập nhật best
cặp (0,3): 2 + 3 - 1 = 4 | best = 5
cặp (0,4): 2 + 1 - 0 = 3 | best = 5
cặp (1,2): 3 + 3 - 1 = 5 | best = 5
cặp (1,3): 3 + 3 - 1 = 5 | best = 5
cặp (1,4): 3 + 1 - 0 = 4 | best = 5
cặp (2,3): 3 + 3 - 1 = 5 | best = 5
cặp (2,4): 3 + 1 - 1 = 3 | best = 5
cặp (3,4): 3 + 1 - 0 = 4 | best = 5
Kết quả: 5 đạt tại (0, 2)


Trace Cách 3 (sắp xếp + cắt tỉa) trên **Ví dụ 3** — để thấy vì sao số cặp phải xét giảm mạnh:

In [16]:
def trace_sorted(n, roads):
    deg = degrees(n, roads)
    adj = [set() for _ in range(n)]
    for a, b in roads:
        adj[a].add(b); adj[b].add(a)
    order = sorted(range(n), key=lambda v: -deg[v])
    print("order =", order, "bậc:", [deg[v] for v in order])
    best = 0
    for i in range(n - 1):
        u = order[i]
        if deg[u] + deg[order[i + 1]] <= best:
            print(f"i={i}: u={u}, cận trên {deg[u]}+{deg[order[i+1]]} <= best={best} -> DỪNG"); break
        for j in range(i + 1, n):
            v = order[j]; s = deg[u] + deg[v]
            if s <= best:
                print(f"   ({u},{v}): tổng bậc {s} <= best={best} -> break vòng trong"); break
            if v in adj[u]:
                best = max(best, s - 1); print(f"   ({u},{v}): nối nhau, rank={s-1}, best={best}")
            else:
                best = s; print(f"   ({u},{v}): KHÔNG nối, rank={s}, best={best} -> break (cận trên)"); break
    print("Kết quả:", best)
    return best

assert trace_sorted(*examples[2][:2]) == 5

order = [2, 1, 5, 0, 3, 4, 6, 7] bậc: [3, 2, 2, 1, 1, 1, 1, 1]
   (2,1): nối nhau, rank=4, best=4
   (2,5): KHÔNG nối, rank=5, best=5 -> break (cận trên)
i=1: u=1, cận trên 2+2 <= best=5 -> DỪNG
Kết quả: 5


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** (bậc + tập kề, duyệt mọi cặp) — ngắn, dễ kiểm chứng, `O(n² + m)` dư sức với `n ≤ 100`.

In [17]:
class Solution:
    def maximalNetworkRank(self, n: int, roads: List[List[int]]) -> int:
        deg = [0] * n
        adj = [set() for _ in range(n)]
        for a, b in roads:
            deg[a] += 1
            deg[b] += 1
            adj[a].add(b)
            adj[b].add(a)
        best = 0
        for i in range(n):
            for j in range(i + 1, n):
                rank = deg[i] + deg[j] - (1 if j in adj[i] else 0)
                if rank > best:
                    best = rank
        return best

for n, roads, exp in examples:
    got = Solution().maximalNetworkRank(n, roads)
    print(got, exp); assert got == exp

4 4
5 5
5 5


## 10. Vì sao thuật toán đúng

**Bổ đề (công thức rank).** Gọi `A_i` = tập đường chạm `i`. Network rank của `(i, j)` theo định nghĩa là `|A_i ∪ A_j|`. Theo nguyên lý bao hàm – loại trừ: `|A_i ∪ A_j| = |A_i| + |A_j| − |A_i ∩ A_j|`. Một đường thuộc `A_i ∩ A_j` phải có hai đầu là `i` và `j` (một đường chỉ có hai đầu và `i ≠ j`), và đề đảm bảo có nhiều nhất một đường như vậy ⇒ `|A_i ∩ A_j| = connected(i, j) ∈ {0,1}`. ∎

**Cách 2 đúng** vì nó tính chính xác công thức trên cho **mọi** cặp và lấy max.

**Cách 3 đúng** vì: (1) `rank(u, v) ≤ deg[u] + deg[v]`; (2) với `u` cố định, các `v` sau trong `order` có bậc không tăng nên tổng bậc không tăng — khi tổng `≤ best` hoặc vừa gặp cặp không nối (đạt đúng cận trên), không `v` nào phía sau làm tốt hơn; (3) tương tự, cận trên `deg[u] + deg[order[i+1]]` bao mọi cặp có phần tử nhỏ nhất (theo order) từ `i` trở đi.

**Cách 4 đúng** theo lập luận ở mục 7.3: mọi cặp ngoài nhóm ứng viên có rank `≤` giá trị "−1" của nhóm ứng viên, nên chỉ cần biết nhóm ứng viên có cặp không nối hay không — đếm số đường trong nhóm và so với số cặp tối đa.

## 11. Phân tích Time / Space Complexity

Gọi `n` = số thành phố (`≤ 100`), `m` = số đường (`≤ n(n−1)/2 = 4950`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn theo định nghĩa | `O(n²·m)` (tệ nhất `O(n⁴)`) | `O(1)` |
| 2. Bậc + kề, mọi cặp | `O(n² + m)` | `O(n + m)` / `O(n²)` nếu dùng ma trận |
| 3. Sắp xếp + cắt tỉa | `O(n log n + m)` + số cặp xét; tệ nhất `O(n²)` | `O(n + m)` |
| 4. Phân tích trường hợp | `O(n + m)` | `O(n)` |

Vì `m ≤ n²/2`, với giới hạn đề mọi cách từ 2 trở đi đều chỉ vài mili-giây. Đo nhanh trên đồ thị đầy đủ `n = 100` (tệ nhất cho Cách 3 vì mọi cặp đều nối):

In [18]:
complete = [[i, j] for i, j in combinations(range(100), 2)]
for name, f in [("Cách 2", rank_pairs), ("Cách 3", rank_sorted), ("Cách 4", rank_linear)]:
    t = time.perf_counter(); r = f(100, complete); dt = time.perf_counter() - t
    print(f"{name}: {r}  ({dt*1000:.2f} ms)")
assert rank_pairs(100, complete) == 2 * 99 - 1

Cách 2: 197  (0.52 ms)
Cách 3: 197  (1.09 ms)
Cách 4: 197  (0.41 ms)


## 12. Bộ test

Gồm: ví dụ đề, edge case (không có đường, `n = 2`, đồ thị đầy đủ, hình sao, hai trung tâm nối/không nối), phản ví dụ cho các heuristic sai, và đối chiếu ngẫu nhiên có seed cố định giữa mọi cách.

In [19]:
ALL = {
    "brute": rank_bruteforce,
    "pairs": rank_pairs,
    "sorted": rank_sorted,
    "linear": rank_linear,
    "SolPairs": lambda n, r: SolutionPairs().maximalNetworkRank(n, r),
    "SolSorted": lambda n, r: SolutionSorted().maximalNetworkRank(n, r),
    "SolLinear": lambda n, r: SolutionLinear().maximalNetworkRank(n, r),
    "submit": lambda n, r: Solution().maximalNetworkRank(n, r),
}

def run_all(n, roads, skip_brute=False):
    res = {k: f(n, roads) for k, f in ALL.items() if not (skip_brute and k == "brute")}
    assert len(set(res.values())) == 1, res
    return next(iter(res.values()))

for n, roads, exp in examples:
    assert run_all(n, roads) == exp

edge = [
    (2, [], 0),                                       # không có đường
    (2, [[0, 1]], 1),                                 # một đường, hai đầu là chính cặp
    (5, [], 0),
    (5, [[0, 1], [0, 2], [0, 3], [0, 4]], 4),         # hình sao: tâm + lá = 4 + 1 - 1
    (6, [[0, 2], [0, 3], [1, 4], [1, 5]], 4),         # hai tâm KHÔNG nối: 2 + 2
    (6, [[0, 1], [0, 2], [0, 3], [1, 4], [1, 5]], 5), # hai tâm nối: 3 + 3 - 1
    (4, [[0, 1], [1, 2], [2, 3], [3, 0]], 4),         # chu trình 4: cặp đối diện 2 + 2
    (3, [[0, 1], [1, 2], [0, 2]], 3),                 # tam giác: 2 + 2 - 1
]
for n, roads, exp in edge:
    assert run_all(n, roads) == exp, (n, roads)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [20]:
# Ngẫu nhiên, seed cố định, đúng ràng buộc: 2 <= n, không cạnh lặp, không khuyên
rng = random.Random(1615)
for it in range(500):
    n = rng.randint(2, 12)
    pairs = list(combinations(range(n), 2))
    m = rng.randint(0, len(pairs))
    roads = [list(p) if rng.random() < 0.5 else [p[1], p[0]] for p in rng.sample(pairs, m)]
    run_all(n, roads)
print("500 test nhỏ (có brute force): OK")

for it in range(60):
    n = rng.randint(2, 100)
    pairs = list(combinations(range(n), 2))
    m = rng.randint(0, len(pairs))
    roads = [list(p) for p in rng.sample(pairs, m)]
    run_all(n, roads, skip_brute=True)
print("60 test lớn (n <= 100, bỏ brute): OK")

500 test nhỏ (có brute force): OK
60 test lớn (n <= 100, bỏ brute): OK


In [21]:
# Đồ thị "đều" (mọi đỉnh cùng bậc) — tình huống dễ làm sai Cách 3/4
def regular_cycle_power(n, k):
    """Mỗi đỉnh nối k đỉnh kế tiếp theo vòng tròn -> bậc 2k (n > 2k)."""
    E = set()
    for v in range(n):
        for t in range(1, k + 1):
            a, b = v, (v + t) % n
            E.add((min(a, b), max(a, b)))
    return [list(e) for e in E]

for n, k in [(5, 1), (5, 2), (7, 3), (9, 2), (10, 4)]:
    r = regular_cycle_power(n, k)
    print(f"n={n}, bậc={2*k}: rank={run_all(n, r)}")

n=5, bậc=2: rank=4
n=5, bậc=4: rank=7
n=7, bậc=6: rank=11
n=9, bậc=4: rank=8
n=10, bậc=8: rank=16


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ 1 khi hai thành phố nối nhau

In [22]:
def wrong_no_minus(n, roads):
    deg = degrees(n, roads)
    return max(deg[i] + deg[j] for i in range(n) for j in range(i + 1, n))

n, roads, exp = examples[0]
print("sai :", wrong_no_minus(n, roads), "| đúng:", Solution().maximalNetworkRank(n, roads))

sai : 5 | đúng: 4


**Vì sao sai:** đường `[0,1]` nằm trong cả `deg[0]` và `deg[1]` nên bị đếm hai lần; đề yêu cầu đếm một lần. **Sửa:** trừ `connected(i, j)`.

### Lỗi 2 — Chỉ lấy hai thành phố có bậc cao nhất

In [23]:
def wrong_top2(n, roads):
    deg = degrees(n, roads)
    adj = [set() for _ in range(n)]
    for a, b in roads:
        adj[a].add(b); adj[b].add(a)
    u, v = sorted(range(n), key=lambda x: -deg[x])[:2]
    return deg[u] + deg[v] - (v in adj[u])

cyc = [[0, 1], [1, 2], [2, 3], [3, 0]]
print("chu trình 4 đỉnh:", cyc)
print("sai :", wrong_top2(4, cyc), "| đúng:", Solution().maximalNetworkRank(4, cyc))

chu trình 4 đỉnh: [[0, 1], [1, 2], [2, 3], [3, 0]]
sai : 3 | đúng: 4


**Vì sao sai:** mọi đỉnh đều bậc 2; hai đỉnh "đầu tiên" là 0 và 1 lại **nối nhau** (rank 3), trong khi cặp đối diện (0, 2) không nối có rank 4. Khi có nhiều đỉnh cùng bậc cao nhất phải xét **tất cả** cặp trong nhóm đó (Cách 4) hoặc duyệt mọi cặp. (Phản ví dụ tự dựng — đã kiểm chứng bằng brute force ở mục 12; đối chiếu lại đề gốc.)

### Lỗi 3 — Chỉ xét các cặp **có đường nối** (duyệt theo `roads`)

In [24]:
def wrong_only_edges(n, roads):
    deg = degrees(n, roads)
    return max((deg[a] + deg[b] - 1 for a, b in roads), default=0)

n, roads, exp = examples[2]
print("sai :", wrong_only_edges(n, roads), "| đúng:", Solution().maximalNetworkRank(n, roads))

sai : 4 | đúng: 5


**Vì sao sai:** đề hỏi mọi cặp **thành phố**, không chỉ cặp có đường. Ví dụ 3: cặp tốt nhất (2, 5) **không** nối nhau và nằm ở hai thành phần liên thông khác nhau. **Sửa:** duyệt mọi cặp `i < j`.

### Lỗi 4 — Cho phép `i == j` (vòng trong bắt đầu từ 0)

In [25]:
def wrong_same_city(n, roads):
    deg = degrees(n, roads)
    adj = [set() for _ in range(n)]
    for a, b in roads:
        adj[a].add(b); adj[b].add(a)
    return max(deg[i] + deg[j] - (j in adj[i]) for i in range(n) for j in range(n))

n, roads, exp = examples[0]
print("sai :", wrong_same_city(n, roads), "| đúng:", Solution().maximalNetworkRank(n, roads))

sai : 6 | đúng: 4


**Vì sao sai:** với `i == j`, `j in adj[i]` là `False` (không có khuyên), nên công thức cho `2·deg[i]` — đếm thành phố 1 "hai lần" ra 6. Đề yêu cầu hai thành phố **khác nhau**. **Sửa:** `for j in range(i + 1, n)`.

### Lỗi 5 — Cách 4: so sánh với `|M|` thay vì số **cặp** trong `M`

In [26]:
def wrong_linear(n, roads):
    deg = degrees(n, roads)
    d1 = max(deg)
    M = [v for v in range(n) if deg[v] == d1]
    k = len(M)
    if k >= 2:
        e = sum(1 for a, b in roads if deg[a] == d1 and deg[b] == d1)
        return 2 * d1 if e < k else 2 * d1 - 1         # SAI: phải là k*(k-1)//2
    return rank_linear(n, roads)

print("chu trình 4:", "sai :", wrong_linear(4, cyc), "| đúng:", Solution().maximalNetworkRank(4, cyc))

chu trình 4: sai : 3 | đúng: 4


**Vì sao sai:** chu trình 4 đỉnh có `k = 4` đỉnh bậc max và `e = 4` đường giữa chúng; `e < k` sai nên trả `3`. Nhưng số cặp là `4·3/2 = 6 > 4` ⇒ còn cặp không nối ⇒ đáp án `4`. **Sửa:** so với `k·(k−1)/2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số cạnh chạm **ít nhất một** trong hai đỉnh" ⇒ **bao hàm – loại trừ**: `deg[a] + deg[b] − |cạnh chung|`.
- Đồ thị **đơn** (không cạnh lặp, không khuyên) ⇒ phần chung chỉ là 0 hoặc 1.
- `n ≤ 100` (hoặc `≤ 1000`) ⇒ duyệt mọi cặp `O(n²)` là đủ; đừng cố tối ưu sớm.
- Cần kiểm tra kề nhanh: `set` cho mỗi đỉnh, hoặc ma trận khi `n` nhỏ.
- Thấy "lấy max của tổng hai đại lượng có điều kiện phụ" ⇒ thử **sắp xếp + cắt tỉa** hoặc **phân tích top-1/top-2**.
- Luôn tự hỏi: hai đỉnh **không liên thông** có được tính không? (Có!)

## 15. Biến thể của bài

1. **Trả về cặp thành phố** đạt max (thay vì giá trị) — xem Bài tập 4.
2. **Network rank của bộ ba** thành phố: `deg[a] + deg[b] + deg[c] − (số đường giữa chúng)` — Bài tập 5.
3. **Đồ thị có cạnh lặp (multigraph):** phần chung là **số** đường giữa `a` và `b` (có thể > 1) ⇒ dùng `Counter` cho cặp cạnh.
4. **`n` lớn tới `10⁵`:** bắt buộc dùng Cách 4 `O(n + m)`.
5. Bài liên quan: **1557 Minimum Number of Vertices to Reach All Nodes** (bậc vào), **997 Find the Town Judge** (bậc vào/ra), **1761 Minimum Degree of a Connected Trio in a Graph** (bậc của bộ ba — rất gần Biến thể 2).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mỗi cặp đếm lại mọi đường | `O(n²·m)` | `O(1)` | Kiểm chứng, test nhỏ |
| 2. Bậc + kề | `deg[i] + deg[j] − kề` cho mọi cặp | `O(n² + m)` | `O(n + m)` | **Bản nộp khuyên dùng** với `n ≤ 100` |
| 3. Sắp xếp + cắt tỉa | Duyệt theo bậc giảm, dừng sớm | `O(n log n + m)` ~ `O(n²)` tệ nhất | `O(n + m)` | Đồ thị thưa, muốn nhanh trong thực tế |
| 4. Phân tích top-1/top-2 | Đếm cạnh trong nhóm ứng viên | `O(n + m)` | `O(n)` | `n` rất lớn; phỏng vấn hỏi follow-up |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `n=3, roads=[]`; (b) `n=3, roads=[[0,1],[1,2]]`; (c) `n=4, roads=[[0,1],[2,3]]`; (d) `n=4` đồ thị đầy đủ.

**Bài 2 (tính tay):** Với `n = 6, roads = [[0,1],[0,2],[0,3],[1,2],[4,5]]`, hãy lập bảng bậc, tìm cặp có rank lớn nhất và giá trị của nó.

**Bài 3 (điền TODO):** Hoàn thiện `network_rank(n, roads, a, b)` trả về network rank của **một** cặp cụ thể, trong `O(m)`.

**Bài 4 (tự viết lại):** Viết `best_pair(n, roads)` trả về cặp `(i, j)` với `i < j` có rank lớn nhất; nếu nhiều cặp hòa, chọn cặp nhỏ nhất theo thứ tự từ điển.

**Bài 5 (mini challenge):** Viết `max_trio_rank(n, roads)` trả về giá trị lớn nhất của số đường chạm ít nhất một trong **ba** thành phố khác nhau (`n ≥ 3`).

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def network_rank(n, roads, a, b):
    cnt = 0
    for x, y in roads:
        # TODO: tăng cnt nếu đường (x, y) chạm a hoặc b
        pass
    return cnt

In [28]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_pair(n, roads):
    # TODO: bậc + tập kề, duyệt i < j, chỉ cập nhật khi rank > best (giữ cặp đầu tiên)
    pass

def max_trio_rank(n, roads):
    # TODO: deg[a] + deg[b] + deg[c] - (số đường giữa a, b, c)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính bậc. Mức 2: nhớ trừ 1 nếu nối. Mức 3: (a) `0`; (b) `2` (cặp (0,1): 1+2−1; cặp (0,2): 1+1−0 = 2); (c) `2` (cặp (0,2) không nối: 1+1); (d) `2·3 − 1 = 5`.

**Bài 2.** Mức 1: `deg = [3, 2, 2, 1, 1, 1]`. Mức 2: cặp (0, x) với x ∉ {1,2,3} cho `3 + 1 = 4`; (0,1) cho `3 + 2 − 1 = 4`. Mức 3: đáp án `4`, cặp nhỏ nhất theo từ điển là `(0, 1)`.

**Bài 3.** Mức 1: điều kiện "chạm `a` hoặc `b`". Mức 2: `if x in (a, b) or y in (a, b)`. Mức 3: đường `[a, b]` thỏa điều kiện một lần, tự nhiên đếm đúng một lần.

**Bài 4.** Mức 1: như Cách 2. Mức 2: duyệt `i` tăng, `j` tăng ⇒ thứ tự từ điển. Mức 3: chỉ cập nhật khi `rank > best` (dấu `>` chặt).

**Bài 5.** Mức 1: bao hàm – loại trừ cho 3 tập: `|A∪B∪C| = Σ|A| − Σ|A∩B| + |A∩B∩C|`. Mức 2: `|A∩B∩C| = 0` (một đường chỉ có 2 đầu). Mức 3: duyệt `combinations(range(n), 3)` — `O(n³)` ≈ 1,6·10⁵ bộ ba với `n = 100`, ổn.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, 2
K4 = [[i, j] for i, j in combinations(range(4), 2)]
b1 = [Solution().maximalNetworkRank(3, []), Solution().maximalNetworkRank(3, [[0, 1], [1, 2]]),
      Solution().maximalNetworkRank(4, [[0, 1], [2, 3]]), Solution().maximalNetworkRank(4, K4)]
print("Bài 1:", b1); assert b1 == [0, 2, 2, 5]
roads2 = [[0, 1], [0, 2], [0, 3], [1, 2], [4, 5]]
print("Bài 2: deg =", degrees(6, roads2), "| đáp án =", Solution().maximalNetworkRank(6, roads2))
assert degrees(6, roads2) == [3, 2, 2, 1, 1, 1] and Solution().maximalNetworkRank(6, roads2) == 4

Bài 1: [0, 2, 2, 5]
Bài 2: deg = [3, 2, 2, 1, 1, 1] | đáp án = 4


In [30]:
# Bài 3, 4 — đáp án
def network_rank(n, roads, a, b):
    cnt = 0
    for x, y in roads:
        if x in (a, b) or y in (a, b):
            cnt += 1
    return cnt

def best_pair(n, roads):
    deg = degrees(n, roads)
    adj = [set() for _ in range(n)]
    for a, b in roads:
        adj[a].add(b); adj[b].add(a)
    best, arg = -1, None
    for i in range(n):
        for j in range(i + 1, n):
            r = deg[i] + deg[j] - (j in adj[i])
            if r > best:
                best, arg = r, (i, j)
    return arg

print("Bài 4 trên Bài 2:", best_pair(6, roads2)); assert best_pair(6, roads2) == (0, 1)
rng = random.Random(42)
for _ in range(200):
    n = rng.randint(2, 9)
    pairs = list(combinations(range(n), 2))
    roads = [list(p) for p in rng.sample(pairs, rng.randint(0, len(pairs)))]
    ranks = {(i, j): network_rank(n, roads, i, j) for i, j in pairs}
    mx = max(ranks.values())
    assert mx == Solution().maximalNetworkRank(n, roads)
    assert best_pair(n, roads) == min(p for p, r in ranks.items() if r == mx)
print("Bài 3, 4: OK")

Bài 4 trên Bài 2: (0, 1)
Bài 3, 4: OK


In [31]:
# Bài 5 — đáp án
def max_trio_rank(n, roads):
    deg = degrees(n, roads)
    E = {(min(a, b), max(a, b)) for a, b in roads}
    best = 0
    for a, b, c in combinations(range(n), 3):
        inner = ((a, b) in E) + ((a, c) in E) + ((b, c) in E)
        best = max(best, deg[a] + deg[b] + deg[c] - inner)
    return best

def trio_brute(n, roads):
    return max(sum(1 for x, y in roads if x in t or y in t) for t in combinations(range(n), 3))

rng = random.Random(3)
for _ in range(150):
    n = rng.randint(3, 8)
    pairs = list(combinations(range(n), 2))
    roads = [list(p) for p in rng.sample(pairs, rng.randint(0, len(pairs)))]
    assert max_trio_rank(n, roads) == trio_brute(n, roads)
print("Bài 5 trên Ví dụ 2:", max_trio_rank(*examples[1][:2]), "| kiểm chứng ngẫu nhiên: OK")

Bài 5 trên Ví dụ 2: 6 | kiểm chứng ngẫu nhiên: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "bậc + kiểm tra kề + duyệt cặp":**

```python
deg = [0] * n
adj = [set() for _ in range(n)]
for a, b in roads:
    deg[a] += 1; deg[b] += 1
    adj[a].add(b); adj[b].add(a)
best = 0
for i in range(n):
    for j in range(i + 1, n):          # i < j: hai thành phố khác nhau
        best = max(best, deg[i] + deg[j] - (j in adj[i]))
return best
```

**Checklist:**
- [ ] Trừ 1 khi hai thành phố **nối trực tiếp** (đường chung chỉ đếm một lần)?
- [ ] Duyệt **mọi** cặp, kể cả cặp không nối / khác thành phần liên thông?
- [ ] `j` bắt đầu từ `i + 1` (không ghép một thành phố với chính nó)?
- [ ] Tăng bậc **cả hai** đầu của mỗi đường?
- [ ] `roads` rỗng ⇒ trả `0` (không lỗi `max` của dãy rỗng — `n ≥ 2` nên luôn có cặp)?
- [ ] Nếu tối ưu bằng top-bậc: đã xét **mọi** đỉnh cùng bậc max, so với `k(k−1)/2`?
- [ ] Đã test: không đường, hình sao, chu trình 4, đồ thị đầy đủ, ngẫu nhiên đối chiếu brute force?